``` text
Pipeline
imports → load data → explore → preprocessing data → split → TF-IDF → model → evaluate →  embeding → model → evaluate.
```

**“What task am I trying to do, and which tool/library provides it?”**
| What do I need to do?               | Tool I need      | Import                                                              |
| ----------------------------------- | ---------------- | ------------------------------------------------------------------- |
| Regular expressions / text cleaning | `re`             | `import re`                                                         |
| Arrays, vectors, math               | NumPy            | `import numpy as np`                                                |
| Count labels                        | `Counter`        | `from collections import Counter`                                   |
| Load 20 Newsgroups                  | sklearn datasets | `from sklearn.datasets import fetch_20newsgroups`                   |
| Split train/test                    | sklearn          | `from sklearn.model_selection import train_test_split`              |
| Train Logistic Regression           | sklearn          | `from sklearn.linear_model import LogisticRegression`               |
| Calculate accuracy/report           | sklearn          | `from sklearn.metrics import accuracy_score, classification_report` |
| Create TF-IDF                       | sklearn          | `from sklearn.feature_extraction.text import TfidfVectorizer`       |
| Load GloVe                          | gensim           | `import gensim.downloader as api`                                   |
| Tokenize/lemmatize                  | spaCy            | `import spacy`                                                      |


----
```` text 
NumPy       → numbers / arrays
pandas      → tables / DataFrames

scikit-learn
   ↓
datasets    → datasets
model_selection → splitting
linear_model    → models
metrics         → evaluation
feature_extraction → TF-IDF

spaCy       → NLP preprocessing
gensim      → embeddings



In [15]:
# I need to load the dataset
# What function did we use?
# import data 
import pandas as pd 
# Arrays, vectors, math
import numpy as np
# Regular expressions / text cleaning
import re
# count labels tools we need is Counter 
from collections import Counter
# fetch_20newsgroups import dataset
from sklearn.datasets import fetch_20newsgroups
# data need to split 
from sklearn.model_selection import train_test_split
# data to to be modeled
from sklearn.linear_model import LogisticRegression
# evaluation and matric
from sklearn.metrics import accuracy_score, classification_report
# feature_extraction
from sklearn.feature_extraction.text import TfidfVectorizer
# Load GloVe tools gensim 
import gensim.downloader as api
# import tekenoize/lemmatize
import spacy
# Make results reproducible
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

### Load data and and exploring the data 
``` text 
TODAY

1. LOAD DATASET ↓ Choose 4 categories ↓ Load 20 Newsgroups ↓
                                             Create:
                                                texts
                                                labels
                                                target_names

   2. EXPLORE DATA
      ↓
      How many documents?
      ↓
      What are the categories?
      ↓
      What are the label IDs?
      ↓
      Look at example documents
      ↓
      Check class distribution

3. PREPROCESSING
   ↓
   Load spaCy
   ↓
   Try to build preprocess_text()
   ↓
   Test it on ONE document

In [27]:
# 1. Choose categories
categories = [
    "rec.sport.baseball",
    "rec.sport.hockey",
    "sci.med",
    "sci.space",
]
# 2- loading the data 
newsgroups = fetch_20newsgroups(subset= 'all',
                                categories=categories,
                                remove =  ('headers', 'footers', 'quotes'), )
# 3- Create text , labels , target names
#supervised ML, you know from your previous learning that you need:
# X → input/features
# y → target
# for text classification
# X = documents
# y = categories
# Extract documents : I need the actual text the will become my input 
texts = newsgroups.data
# now we need target 
labels = newsgroups.target # these are label in the form of number  and are not readable by human
# we need the humane read the labels
target_names = newsgroups.target_names

# 4- Now should check the data 
# Rule: Never Load data and immediately start modeling , Inspect it first.
# How much data did I load? len()
print ("number of documents:", len(texts))
# Did I get the correct categories?
print('categories:', target_names)
# what labels actually exists, there are many , contains thousands ,we do not need to know all of them 
# we need the unique value and we use set to get unique labales and order them to make it clean
print ('label IDs:', sorted(set(labels)))

# in conclusion: 
#supervised learning → need X and y
# multiple values → list
# load data → inspect it
# check size
# check classes
#check labels
# You are allowed to look up:

# fetch_20newsgroups
#  subset="all" remove=(...)
# .data
# .target
# .target_names



number of documents: 3970
categories: ['rec.sport.baseball', 'rec.sport.hockey', 'sci.med', 'sci.space']
label IDs: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


=======================================================

**look at one real document from each category**

`we have 4 documents and we want to see one actual text from each document.`

**Check how many documents belong to each category**
 

``` text
I want one example from EACH category
              ↓
Need to repeat across categories
              ↓
           FOR loop
              ↓
Find documents belonging to current category
              ↓
Take the first matching document
              ↓
Save its index
              ↓
Move to next category

```

In [20]:
example_indices =[] # an empty list to store the index of one example from each category.
# So we need an empty list because we're building something new.
# what am I repeating : class_id  for 4 use range
for class_id in range(len(target_names)):
    idx = np.where(labels == class_id)[0][0]
    example_indices.append(idx)
# Print snippets
# we want to print class_id and idx and use enumerate to give both
for class_id, idx in enumerate(example_indices):
    print("=" * 80)
    print("Category:", target_names[class_id])
    print("Document index:", idx)
    print("-" * 80)
    print(texts[idx][:600])  # first 600 characters
    print()



Category: rec.sport.baseball
Document index: 1
--------------------------------------------------------------------------------
Name            Pos   AB    H    2B    3B    HR    RBI    RS    SB    E    AVG
------------------------------------------------------------------------------
Boston          OF    12    7                        2     6              .583
Galarraga       1B    28   13     3           1      9     2              .464
Tatum           3B     5    2     1                                       .400
Cole            CF    24    9           1            2     8     2        .375
E. Young        2B    28    9     1     1     1      5    10     5    3   .321
Hayes           3B    25    7     1           2

Category: rec.sport.hockey
Document index: 6
--------------------------------------------------------------------------------
[more about the Messier-Samuelsson incident]
 I agree with Rick that Ulf's cross check wasn't illegal. It was the kind
 of check you see a dozen

In [21]:
# 3. Check class distribution 
# : Do my four categories contain approximately similar amounts of data?
counts = Counter(labels) # counter counts how many times each label occurs
print("Class distribution:")
for class_id, idx in sorted(counts.items()):
   print(f"  {class_id}: {target_names[class_id]:20s} -> {counts} documents")





Class distribution:
  0: rec.sport.baseball   -> Counter({np.int64(1): 999, np.int64(0): 994, np.int64(2): 990, np.int64(3): 987}) documents
  1: rec.sport.hockey     -> Counter({np.int64(1): 999, np.int64(0): 994, np.int64(2): 990, np.int64(3): 987}) documents
  2: sci.med              -> Counter({np.int64(1): 999, np.int64(0): 994, np.int64(2): 990, np.int64(3): 987}) documents
  3: sci.space            -> Counter({np.int64(1): 999, np.int64(0): 994, np.int64(2): 990, np.int64(3): 987}) documents


```text
DATA LOADED
    ↓
What does my text actually look like?
    ↓
Look at one document from each class
    ↓
Do I have enough examples in each class?
    ↓
Count documents per class
    ↓
DATA EXPLORATION COMPLETE
    ↓
TODAY'S NEXT MAJOR STEP:
PREPROCESSING


---

## Preprocessing: Cleaning Our Text

Before we can feed text to a machine learning model, we need to **clean it up**. This is called **preprocessing**.
###  Our Preprocessing Steps:

| Step | What it does | Example |
|------|-------------|--------|
| 1. **Lowercasing** | Makes everything lowercase | "HELLO" → "hello" |
| 2. **Tokenization** | Splits text into words | "hello world" → ["hello", "world"] |
| 3. **Remove punctuation** | Removes !, ?, . etc. | "hello!" → "hello" |
| 4. **Lemmatization** | Converts words to base form | "running" → "run" |

### What is Lemmatization? 🤔

**Lemmatization** converts words to their "dictionary form" (called the *lemma*):



``` text 
                     spaCy English model
                           ↓
text ───────────────→     nlp
                           ↓
                    processed document

In [26]:
# Preprocessing Step 1 — Load the spaCy model
# spacy has different trained language model we download a short part of it 
# we need to download english model 
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

# `ner means Named Entity Recognition. 
# It can recognize things such as people, organizations, and locations.
# `parser` performs dependency parsing, analyzing grammatical relationships between words.
print("spaCy pipeline components:", nlp.pipe_names)

spaCy pipeline components: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']


---
### 3.2 Define a preprocessing function
This function converts raw text into a list of **clean lemma tokens**.


In [29]:
def preprocess_text(text):

    # lowercase
    text = text.lower()
    # process with spaCy
    doc = nlp(text)
    # create empty list
    tokens =[]
    # loop through tokens
    for token in doc:
        # skip spaces/punctuation
        if token.is_space or token.is_punct:
            continue
        # skip non-alphabetic tokens
        if not token.is_alpha:
            continue
        # get lemma
        lemma =token.lemma_.strip()
        # handle bad lemma
        # spaCy sometimes uses "-PRON-" for pronouns in older models;
        # if that happens, fall back to the original token text.
        if lemma == "-PRON-" or lemma =="":
            lemma =token.text

        # save lemma
        tokens.append(lemma)

    # return cleaned tokens
    return tokens


# The first function retun seperate lemmas, But later TF-IDF wants text in string form.
def tokens_to_string(tokens):
    """Join tokens back into a single string (useful for TF-IDF)."""
    return " ".join(tokens)

In [31]:
# identify sample text 
sample_text = texts[0]
print("ORIGINAL (first 400 chars):")
print(sample_text[:400])
print()

sample_tokens =preprocess_text(sample_text)
print("PREPROCESSED TOKENS (first 40):")
print(sample_tokens[:40])
print()
print( "Number of tokens:", len(sample_tokens))

ORIGINAL (first 400 chars):

Hum, do you enjoy putting words in my mouth? 
Come to Nome and meet some of these miners.. I am not sure how things go down
south in the lower 48 (I used to visit, but), of course to believe the
media/news its going to heck (or just plain crazy). 
Well it seems that alot of Unionist types seem to think that having a job is a
right, and not a priviledge. Right to the same job as your forbearers, S

PREPROCESSED TOKENS (first 40):
['hum', 'do', 'you', 'enjoy', 'put', 'word', 'in', 'my', 'mouth', 'come', 'to', 'nome', 'and', 'meet', 'some', 'of', 'these', 'miner', 'I', 'be', 'not', 'sure', 'how', 'thing', 'go', 'down', 'south', 'in', 'the', 'low', 'I', 'use', 'to', 'visit', 'but', 'of', 'course', 'to', 'believe', 'the']

Number of tokens: 151


In [34]:
## Preprocess the whole dataset
all_tokens =[]
for t in texts:
    clean_tokens =preprocess_text(t)
    all_tokens.append(clean_tokens)
# Join tokens into strings (handy for TF-IDF)
all_text_clean =[]
for tokens in all_tokens:
    all_string_clean =tokens_to_string(tokens)
    all_text_clean.append( all_string_clean)
print("Example cleaned text (first 200 chars):")
print(all_text_clean[0][:200])
print()
print("Number of documents preprocessed:", len(all_text_clean))

Example cleaned text (first 200 chars):
hum do you enjoy put word in my mouth come to nome and meet some of these miner I be not sure how thing go down south in the low I use to visit but of course to believe the medium news its go to heck 

Number of documents preprocessed: 3970


### Train/Test Split: Preparing Data for Learning
## Approach 1: TF-IDF + Logistic Regression